In [ ]:
import glob
import numpy as np, pandas as pd, geopandas as gpd, folium, osmnx as ox
from sklearn.neighbors import BallTree
from saajha.clean import clean_positions
from saajha.geo import haversine_km

R = 6371000  # Earth radius in metres
stops = pd.read_csv("../data/raw/delhi_bus_gtfs/stops.txt")
f = sorted(glob.glob("../data/raw/bus_positions_*.csv"))[-1]
pos = clean_positions(pd.read_csv(f, dtype={"route_id": str}))
hours = (pos.timestamp.max() - pos.timestamp.min()) / 3600
print(len(stops), "bus stops |", pos.vehicle_id.nunique(), "buses over", round(hours, 2), "hours")

In [ ]:
raw = pd.read_csv("../data/raw/DELHI_METRO_DATA.csv")
stations = (raw.rename(columns={"Station": "name", "Latitude": "lat", "Longitude": "lon"})
               .dropna(subset=["lat", "lon"])
               .groupby("name", as_index=False)[["lat", "lon"]].mean())

# 1. keep stations inside Delhi (the bus feed only covers Delhi)
delhi = ox.geocode_to_gdf("Delhi, India").geometry.iloc[0]
inside = gpd.GeoSeries(gpd.points_from_xy(stations.lon, stations.lat)).within(delhi).values
stations = stations[inside].reset_index(drop=True)

# 2. drop placeholder coordinates shared by several stations
stations = stations[~stations.duplicated(["lat", "lon"], keep=False)].reset_index(drop=True)

# 3. drop stations far from a bus stop named after them
metro_stops = stops[stops.stop_name.str.contains("metro", case=False, na=False)]
def check_m(row):
    key = row["name"].replace("†", "").strip().lower()
    m = metro_stops[metro_stops.stop_name.str.lower().str.contains(key, regex=False)]
    return np.nan if m.empty else haversine_km(row.lat, row.lon, m.stop_lat, m.stop_lon).min() * 1000
stations["check_m"] = stations.apply(check_m, axis=1)
stations = stations[~(stations.check_m > 800)].reset_index(drop=True)

# 4. manual fixes, verified against Wikipedia / MetroGo
fixes = {
    "Chawri Bazar":         (28.649600, 77.226200),
    "Shalimar Bagh":        (28.701819, 77.165191),
    "Chirag Delhi":         (28.541229, 77.229385),
    "Naraina Vihar":        (28.624125, 77.136488),
    "Kirti Nagar":          (28.655100, 77.151500),
    "Munirka":              (28.557898, 77.173985),
    "Rajouri Garden":       (28.649000, 77.122600),
    "Mayur Vihar -I":       (28.604134, 77.289526),
    "Mayur Vihar Pocket I": (28.606598, 77.296326),
    "Sector 28":  None,   # Faridabad
    "Shyam Park": None,   # Ghaziabad
    "Vaishali":   None,   # Ghaziabad
    "Noida Electronic City": None,
}
names = stations.name.str.replace("†", "", regex=False).str.strip()
for name, coord in fixes.items():
    hit = names == name
    if coord is None:
        stations, names = stations[~hit], names[~hit]
    else:
        stations.loc[hit, ["lat", "lon"]] = coord
stations = stations.reset_index(drop=True)
print(len(stations), "validated metro stations")

In [ ]:
st_xy = np.radians(stations[["lat", "lon"]].values)
stop_tree = BallTree(np.radians(stops[["stop_lat", "stop_lon"]].values), metric="haversine")
stations["bus_stops_500m"] = stop_tree.query_radius(st_xy, r=500 / R, count_only=True)

pt_tree = BallTree(np.radians(pos[["lat", "lon"]].values), metric="haversine")
idx = pt_tree.query_radius(st_xy, r=300 / R)
vids, rids = pos.vehicle_id.values, pos.route_id.values
stations["buses_per_hour"] = [len(set(vids[i])) / hours for i in idx]
stations["live_routes"] = [pd.Series(rids[i]).nunique() for i in idx]
stations.describe()

In [ ]:
worst = stations.sort_values(["buses_per_hour", "bus_stops_500m"]).head(20)
worst

In [ ]:
q = stations.buses_per_hour.quantile(0.25)
m = folium.Map(location=[28.63, 77.15], zoom_start=11, tiles="OpenStreetMap")
for _, s in stations.iterrows():
    folium.CircleMarker(
        [s.lat, s.lon], radius=4 + min(s.buses_per_hour, 60) / 6,
        color="red" if s.buses_per_hour <= q else "green", fill=True,
        tooltip=f"{s['name']}: {s.buses_per_hour:.0f} buses/hr, {s.bus_stops_500m} stops within 500 m",
    ).add_to(m)
m.save("../data/processed/last_mile_gaps.html")
m

In [ ]:
live_tree = BallTree(np.radians(pos[["lat", "lon"]].values), metric="haversine")
near = live_tree.query_radius(np.radians(stops[["stop_lat", "stop_lon"]].values), r=150 / R, count_only=True)
stops["live_seen"] = near > 0
print(f"{(~stops.live_seen).mean():.0%} of official bus stops had no live bus within 150 m in the hour")

m2 = folium.Map(location=[28.63, 77.15], zoom_start=11, tiles="OpenStreetMap")
for _, s in stops.iterrows():
    folium.CircleMarker([s.stop_lat, s.stop_lon], radius=2,
                        color="green" if s.live_seen else "red").add_to(m2)
m2.save("../data/processed/stop_coverage.html")

In [ ]:
stop_xy = np.radians(stops[["stop_lat", "stop_lon"]].values)
near_idx = stop_tree.query_radius(np.radians(worst[["lat", "lon"]].values), r=500 / R)
worst = worst.assign(stops_live=[f"{stops.live_seen.values[i].sum()}/{len(i)}" for i in near_idx])
worst[["name", "buses_per_hour", "bus_stops_500m", "stops_live"]]

In [ ]:
import json

q = stations.buses_per_hour.quantile(0.25)
def gap(s):
    if s["name"] == "Indira Gandhi International Airport":
        return "served"                      # buses use terminal bays, not street stops
    if s.bus_stops_500m == 0 and s.buses_per_hour < 5:
        return "none"
    return "missed" if s.buses_per_hour <= q else "served"

features = [{
    "type": "Feature",
    "geometry": {"type": "Point", "coordinates": [round(s.lon, 6), round(s.lat, 6)]},
    "properties": {"name": s["name"].replace("†", "").strip(),
                   "buses_per_hour": round(s.buses_per_hour, 1),
                   "bus_stops_500m": int(s.bus_stops_500m),
                   "live_routes": int(s.live_routes),
                   "gap": gap(s)},
} for _, s in stations.iterrows()]

with open("../docs/stations.geojson", "w", encoding="utf-8") as f:
    json.dump({"type": "FeatureCollection", "features": features}, f)
print(len(features), "stations exported")